In [ ]:
import pandas as pd
from sklearn.metrics import f1_score
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold
from catboost import CatBoostClassifier

---
# Modèle CatBoostClassifier

---
## Validation croisée

In [3]:
df_x_clean = pd.read_csv('../data/x_train_clean.csv')
df_y_clean = pd.read_csv('../data/y_train_clean.csv')
df_x_test_clean = pd.read_csv('../data/x_test_clean.csv')

# les 32 colonnes d'entreprise
company_cols = [f'company_feature_{i}' for i in range(32)]

#regroupe sous un même identifiant les lignes qui ont la meme entreprise
df_x_clean['company_group'] = df_x_clean[company_cols].apply(lambda row: hash(tuple(row)), axis=1)
groups_company = df_x_clean['company_group']


#on va faire une validation croisée en utilisant StratifiedGroupKFold
#le découpage respecte à la fois les groupes d'entités et l'équilibre des classes dans chaque bloc.

X = df_x_clean.drop(columns=['id', 'company_group'])
y = df_y_clean['target']

cv = StratifiedGroupKFold(n_splits=5,random_state=42, shuffle=True)
scores_f1 = []

#entraînement et d'évaluation pli par pli
for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups=groups_company)):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    
    model = CatBoostClassifier(
        loss_function='MultiClass',
        random_seed=42,
        auto_class_weights='Balanced'
    )
    
    model.fit(X_tr, y_tr)
    
    preds = model.predict(X_te)
    score = f1_score(y_te, preds, average='macro')
    scores_f1.append(score)
    
    print(f"Pli {fold + 1} - F1-Macro : {score}")

print(f"\n---> F1-Macro Moyen : {np.mean(scores_f1)}")


Learning rate set to 0.092933
0:	learn: 1.3696201	total: 228ms	remaining: 3m 48s
1:	learn: 1.3538303	total: 262ms	remaining: 2m 10s
2:	learn: 1.3358581	total: 296ms	remaining: 1m 38s
3:	learn: 1.3217602	total: 336ms	remaining: 1m 23s
4:	learn: 1.3077789	total: 368ms	remaining: 1m 13s
5:	learn: 1.2949122	total: 400ms	remaining: 1m 6s
6:	learn: 1.2878437	total: 430ms	remaining: 1m
7:	learn: 1.2778921	total: 466ms	remaining: 57.7s
8:	learn: 1.2646358	total: 502ms	remaining: 55.3s
9:	learn: 1.2548395	total: 535ms	remaining: 53s
10:	learn: 1.2461498	total: 607ms	remaining: 54.6s
11:	learn: 1.2368534	total: 694ms	remaining: 57.1s
12:	learn: 1.2314468	total: 842ms	remaining: 1m 3s
13:	learn: 1.2214491	total: 958ms	remaining: 1m 7s
14:	learn: 1.2153581	total: 1.01s	remaining: 1m 6s
15:	learn: 1.2101215	total: 1.08s	remaining: 1m 6s
16:	learn: 1.2006236	total: 1.12s	remaining: 1m 4s
17:	learn: 1.1927554	total: 1.18s	remaining: 1m 4s
18:	learn: 1.1879220	total: 1.24s	remaining: 1m 4s
19:	learn: 

In [ ]:
#entraînement de ce modèle sur tout le jeu de données d'entraînement
df_x_clean = pd.read_csv('../data/x_train_clean.csv')
df_y_clean = pd.read_csv('../data/y_train_clean.csv')
df_x_test_clean = pd.read_csv('../data/x_test_clean.csv')

X_train = df_x_clean.drop(columns=['id'])
y_train = df_y_clean['target']
X_test = df_x_test_clean.drop(columns=['id'])

model_final1 = CatBoostClassifier(
    loss_function='MultiClass',
    random_seed=42,
    auto_class_weights='Balanced'
)

model_final1.fit(X_train, y_train)

# prédiction sur le jeu de test
predictions = model_final1.predict(X_test)
predictions_flat = predictions.flatten()

# repasser des nombres aux noms de postes
postes = {
    0: "Assistant",
    1: "Executive",
    2: "Manager",
    3: "Director"
}
predictions_labels = [postes[p] for p in predictions_flat]

# créer le fichier
predictions_final_1 = pd.DataFrame({
    'id': df_x_test_clean['id'],
    'position': predictions_labels
})
predictions_final_1.to_csv('../results/predictions_CatBoost_1.csv', index=True)


Learning rate set to 0.093997
0:	learn: 1.3680548	total: 5.4ms	remaining: 5.4s
1:	learn: 1.3511353	total: 11.3ms	remaining: 5.65s
2:	learn: 1.3368567	total: 17.2ms	remaining: 5.73s
3:	learn: 1.3261026	total: 22.2ms	remaining: 5.53s
4:	learn: 1.3140257	total: 27.5ms	remaining: 5.48s
5:	learn: 1.3031218	total: 32.5ms	remaining: 5.39s
6:	learn: 1.2966457	total: 37.7ms	remaining: 5.34s
7:	learn: 1.2864952	total: 42.4ms	remaining: 5.25s
8:	learn: 1.2753383	total: 47.3ms	remaining: 5.21s
9:	learn: 1.2674328	total: 51.9ms	remaining: 5.13s
10:	learn: 1.2603975	total: 56.8ms	remaining: 5.11s
11:	learn: 1.2521757	total: 61.9ms	remaining: 5.09s
12:	learn: 1.2487472	total: 66.4ms	remaining: 5.04s
13:	learn: 1.2405641	total: 71.9ms	remaining: 5.07s
14:	learn: 1.2332194	total: 76.6ms	remaining: 5.03s
15:	learn: 1.2259572	total: 81.4ms	remaining: 5s
16:	learn: 1.2208877	total: 86.2ms	remaining: 4.98s
17:	learn: 1.2155454	total: 91.5ms	remaining: 4.99s
18:	learn: 1.2102160	total: 96.6ms	remaining: 4.9

**Le score de ces prédictions est 0,28636892634872424**

---
## Recherche hyperparamètre

In [ ]:
df_x_clean = pd.read_csv('../data/x_train_clean.csv')
df_y_clean = pd.read_csv('../data/y_train_clean.csv')
df_x_test_clean = pd.read_csv('../data/x_test_clean.csv')

# les 32 colonnes d'entreprise
company_cols = [f'company_feature_{i}' for i in range(32)]

#regroupe sous un même identifiant les lignes qui ont la meme entreprise
df_x_clean['company_group'] = df_x_clean[company_cols].apply(lambda row: hash(tuple(row)), axis=1)
groups_company = df_x_clean['company_group']

#on va faire une validation croisée en utilisant StratifiedGroupKFold
#le découpage respecte à la fois les groupes d'entités et l'équilibre des classes dans chaque bloc.

X = df_x_clean.drop(columns=['id', 'company_group'])
y = df_y_clean['target']


# hyperparamètres à tester
param_test = {
    'iterations': [200, 400, 600, 800],
    'depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.03, 0.08, 0.15],
    'l2_leaf_reg': [1, 3, 5, 10, 20],
    'bagging_temperature': [0.0, 0.5, 1.0],
    'grow_policy': ['SymmetricTree', 'Depthwise']
}



model1 = CatBoostClassifier(
    loss_function='MultiClass',
    random_seed=42,
    auto_class_weights='Balanced'
)

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# recherche aléatoire hyperparamètres
recherche = RandomizedSearchCV(
    estimator=model1,
    param_distributions=param_test,
    n_iter=20,
    scoring='f1_macro',
    cv=cv,
    random_state=42,
    n_jobs=-1
)
recherche.fit(X, y, groups=groups_company)


print(f"Meilleur F1-Macro Réel: {recherche.best_score_:.4f}")
print("\nMeilleurs hyperparamètres trouvés :")
print(recherche.best_params_)

0:	learn: 1.3779253	total: 174ms	remaining: 1m 9s
1:	learn: 1.3689044	total: 305ms	remaining: 1m
0:	learn: 1.3691455	total: 248ms	remaining: 1m 38s
0:	learn: 1.3712230	total: 341ms	remaining: 2m 15s
0:	learn: 1.3721392	total: 343ms	remaining: 2m 16s
0:	learn: 1.3749046	total: 304ms	remaining: 2m 1s
2:	learn: 1.3599771	total: 544ms	remaining: 1m 11s
0:	learn: 1.3782899	total: 362ms	remaining: 2m 24s
0:	learn: 1.3774138	total: 378ms	remaining: 2m 30s
1:	learn: 1.3547237	total: 582ms	remaining: 1m 55s
0:	learn: 1.3735941	total: 430ms	remaining: 2m 51s
3:	learn: 1.3535356	total: 831ms	remaining: 1m 22s
1:	learn: 1.3557567	total: 705ms	remaining: 2m 20s
0:	learn: 1.3794125	total: 427ms	remaining: 2m 50s
1:	learn: 1.3566191	total: 729ms	remaining: 2m 25s
0:	learn: 1.3687929	total: 657ms	remaining: 4m 22s
0:	learn: 1.3748269	total: 642ms	remaining: 2m 7s
0:	learn: 1.3637216	total: 553ms	remaining: 3m 40s
0:	learn: 1.3760760	total: 758ms	remaining: 2m 30s
1:	learn: 1.3605975	total: 912ms	remai

/home/bruyere/Documents/projet_ia/venv/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


114:	learn: 1.1854667	total: 1m 16s	remaining: 3m 8s
399:	learn: 0.6925970	total: 4m 34s	remaining: 4m 34s
382:	learn: 0.7355649	total: 4m 37s	remaining: 5m 2s
726:	learn: 0.5105143	total: 8m 14s	remaining: 49.6s
729:	learn: 0.5179169	total: 7m 56s	remaining: 45.7s
260:	learn: 0.1839435	total: 12m 15s	remaining: 15m 55s
360:	learn: 0.3960875	total: 12m 15s	remaining: 8m 6s
10:	learn: 1.3629971	total: 3.27s	remaining: 56.1s
331:	learn: 0.7409376	total: 4m 4s	remaining: 5m 44s
786:	learn: 0.3246457	total: 8m 41s	remaining: 8.62s
24:	learn: 1.3412846	total: 10s	remaining: 1m 10s
730:	learn: 0.3312377	total: 8m 19s	remaining: 47.1s
273:	learn: 0.1663683	total: 12m 15s	remaining: 14m 35s
142:	learn: 1.1488661	total: 1m 37s	remaining: 2m 54s
11:	learn: 1.3613712	total: 3.4s	remaining: 53.2s
766:	learn: 0.3268139	total: 8m 25s	remaining: 21.8s
367:	learn: 0.4049301	total: 12m 15s	remaining: 7m 43s
30:	learn: 1.3147891	total: 20s	remaining: 3m 58s
135:	learn: 1.1651583	total: 1m 36s	remaining:

On va maintenant entrainer le même modèle avec les meilleurs hyperparamètres trouvés sur l'ensemble des données train 

Meilleurs hyperparamètres:  
'learning_rate': 0.03  
'l2_leaf_reg': 1  
'iterations': 400  
'grow_policy': 'SymmetricTree'  
'depth': 8  
'bagging_temperature': 1.0  

 

In [ ]:
#entraînement du modéle avec les meilleurs hyperparamètres sur tout le jeu de données d'entraînement
df_x_clean = pd.read_csv('../data/x_train_clean.csv')
df_y_clean = pd.read_csv('../data/y_train_clean.csv')
df_x_test_clean = pd.read_csv('../data/x_test_clean.csv')

X_train = df_x_clean.drop(columns=['id'])
y_train = df_y_clean['target']
X_test = df_x_test_clean.drop(columns=['id'])


model_final2 = CatBoostClassifier(
    loss_function='MultiClass',
    random_seed=42,
    auto_class_weights='Balanced',
    iterations=recherche.best_params_['iterations'],
    depth=recherche.best_params_['depth'],
    learning_rate=recherche.best_params_['learning_rate'],
    l2_leaf_reg=recherche.best_params_['l2_leaf_reg'],
    bagging_temperature=recherche.best_params_['bagging_temperature'],
    grow_policy=recherche.best_params_['grow_policy']
)

model_final2.fit(X_train, y_train)

# prédiction sur le jeu de test
predictions = model_final2.predict(X_test)
predictions_flat = predictions.flatten()

# repasser des nombres aux noms de postes
postes = {
    0: "Assistant",
    1: "Executive",
    2: "Manager",
    3: "Director"
}
predictions_labels = [postes[p] for p in predictions_flat]

# créer le fichier
predictions_final_2 = pd.DataFrame({
    'id': df_x_test_clean['id'],
    'position': predictions_labels
})
predictions_final_2.to_csv('../results/predictions_CatBoost_2.csv', index=True)

0:	learn: 1.3785768	total: 27.7ms	remaining: 11.1s
1:	learn: 1.3702185	total: 55ms	remaining: 10.9s
2:	learn: 1.3622378	total: 81.8ms	remaining: 10.8s
3:	learn: 1.3556766	total: 109ms	remaining: 10.7s
4:	learn: 1.3492461	total: 135ms	remaining: 10.7s
5:	learn: 1.3427269	total: 161ms	remaining: 10.6s
6:	learn: 1.3362427	total: 188ms	remaining: 10.5s
7:	learn: 1.3310174	total: 215ms	remaining: 10.5s
8:	learn: 1.3255058	total: 241ms	remaining: 10.5s
9:	learn: 1.3180127	total: 268ms	remaining: 10.5s
10:	learn: 1.3123460	total: 295ms	remaining: 10.4s
11:	learn: 1.3071511	total: 323ms	remaining: 10.4s
12:	learn: 1.3019276	total: 351ms	remaining: 10.4s
13:	learn: 1.2964697	total: 378ms	remaining: 10.4s
14:	learn: 1.2914678	total: 405ms	remaining: 10.4s
15:	learn: 1.2863750	total: 433ms	remaining: 10.4s
16:	learn: 1.2811875	total: 460ms	remaining: 10.4s
17:	learn: 1.2766318	total: 487ms	remaining: 10.3s
18:	learn: 1.2728513	total: 513ms	remaining: 10.3s
19:	learn: 1.2669484	total: 540ms	remain

**Le score de ces prédictions est 0,28636892634872424**